# Project 07 · Kenya against six regional peers

**Benson M. Merita · MeritaAnalytics** · Tools: Python (pandas, NumPy), Matplotlib

**Business question.** How has income per person grown, how stable have prices been, and how does Kenya compare with Uganda, Tanzania, Rwanda, Ethiopia, Nigeria and South Africa? What does that mean for pricing a fixed-rate loan?

**Data.** Real public data: World Bank GDP, population and consumer-price inflation (via Datahub), and Our World in Data CO₂ emissions. Licences: CC BY 4.0 and ODC-PDDL. Credit for the data belongs to the World Bank, Datahub and Our World in Data.

| Step | What it answers |
|---|---|
| 1 | Load the four files |
| 2 | Keep countries only, with a log of dropped rows |
| 3 | Merge on country and year, and reconcile |
| 4 | GDP per person, real growth and inflation volatility |
| 5 | Check outliers, then compare with peers |
| 6 | Charts |
| 7 | What it means for lending |

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, GREEN, GREY, DARK, AMBER = "#1F5FD6", "#2BB38A", "#9CA3AF", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

DATA = Path("../datasets/07-kenya-public-data")
PEERS = {"KEN": "Kenya", "UGA": "Uganda", "TZA": "Tanzania", "RWA": "Rwanda",
         "ETH": "Ethiopia", "NGA": "Nigeria", "ZAF": "South Africa"}

## Step 1 · Load the four files

In [ ]:
gdp = pd.read_csv(DATA / "gdp.csv")
infl = pd.read_csv(DATA / "inflation_consumer.csv")
pop = pd.read_csv(DATA / "population.csv")
co2 = pd.read_csv(DATA / "owid_co2_countries_1960_on.csv")
for name, d in [("gdp", gdp), ("inflation", infl), ("population", pop), ("co2", co2)]:
    year = "Year" if "Year" in d.columns else "year"
    print(f"{name:11s} rows={len(d):6d}  years {d[year].min()}-{d[year].max()}")
gdp.head(3)

## Step 2 · Keep countries only

**What:** World Bank files also contain regions and income groups such as "Sub-Saharan Africa" or "World". Keep only rows whose code is a real country, using the ISO list in the Our World in Data file, and log what was dropped.
**Why:** mixing "World" with countries would double-count in any total or ranking.

In [ ]:
iso = set(co2["iso_code"].dropna())
gdp_c, infl_c, pop_c = (d[d["Country Code"].isin(iso)].copy() for d in (gdp, infl, pop))
log = pd.DataFrame({"source": ["gdp", "inflation", "population"],
                    "rows_in": [len(gdp), len(infl), len(pop)],
                    "rows_kept": [len(gdp_c), len(infl_c), len(pop_c)]})
log["dropped"] = log["rows_in"] - log["rows_kept"]
print("duplicate country-years in GDP:", int(gdp_c.duplicated(["Country Code", "Year"]).sum()))
log

## Step 3 · Merge on country and year, and reconcile

**What:** rename columns to a common layout and merge the four tables on country code and year. `validate="one_to_one"` stops the merge if any country-year appears twice. Then check that no rows were added and that population totals still match the source.
**Why:** a merge that quietly adds or loses rows changes every number after it.

In [ ]:
g = gdp_c.rename(columns={"Country Code": "iso", "Country Name": "country", "Value": "gdp_usd"})[["iso", "country", "Year", "gdp_usd"]]
p = pop_c.rename(columns={"Country Code": "iso", "Value": "population"})[["iso", "Year", "population"]]
i = infl_c.rename(columns={"Country Code": "iso", "Inflation": "inflation"})[["iso", "Year", "inflation"]]
c = co2.rename(columns={"iso_code": "iso", "year": "Year", "population": "pop_owid"})[["iso", "Year", "pop_owid", "co2", "co2_per_capita"]]

m = (g.merge(p, on=["iso", "Year"], how="left", validate="one_to_one")
      .merge(i, on=["iso", "Year"], how="left", validate="one_to_one")
      .merge(c, on=["iso", "Year"], how="left", validate="one_to_one"))
keys = m[["iso", "Year"]].merge(p, on=["iso", "Year"], how="inner")
print("rows: GDP", len(g), "| merged", len(m), "| same:", len(g) == len(m))
print("population total unchanged by the merge:", np.isclose(m["population"].sum(), keys["population"].sum()))
peer_both = m[m["iso"].isin(PEERS)].dropna(subset=["population", "pop_owid"])
ratio = peer_both["population"] / peer_both["pop_owid"]
print(f"World Bank vs OWID population for the 7 countries: ratio {ratio.min():.3f} to {ratio.max():.3f}")

## Step 4 · GDP per person, real growth and inflation

**What:**
- **GDP per person** = GDP ÷ population, in current US dollars.
- **Real** GDP per person removes US price inflation, so growth is not just rising prices. (A simple, transparent choice; it is stated as a limitation.)
- **Five-year blocks** and a **rolling 5-year standard deviation** of Kenya's inflation show how stable prices were.

In [ ]:
m["gdp_pc"] = m["gdp_usd"] / m["population"]
us = infl_c[infl_c["Country Code"] == "USA"].sort_values("Year")
index = (1 + us["Inflation"] / 100).cumprod()
deflator = pd.Series((index / index.iloc[-1]).to_numpy(), index=us["Year"].to_numpy())   # last year = 1.0
m["gdp_pc_real"] = m["gdp_pc"] / m["Year"].map(deflator)
m = m.sort_values(["iso", "Year"]).reset_index(drop=True)
m["real_growth"] = m.groupby("iso")["gdp_pc_real"].pct_change() * 100

k = m[m["iso"] == "KEN"].set_index("Year").copy()
k["block"] = (k.index // 5) * 5
five = k.groupby("block").agg(mean_inflation=("inflation", "mean"), mean_real_growth=("real_growth", "mean")).round(1)
five.index = [f"{b}-{b + 4}" for b in five.index]
five.tail(8)

## Step 5 · Check outliers, then compare with peers

**What:** look for inflation values above 40% since 2000 among the seven countries. Those found are excluded **from the peer comparison only**. Then compute, for 2000–2023: mean inflation, its standard deviation (volatility) and the compound annual growth rate (CAGR) of real GDP per person.
**Why:** one extreme year can dominate an average and distort a ranking. Comparing countries over the same window keeps it fair.

In [ ]:
flagged = m[m["iso"].isin(PEERS) & (m["inflation"] > 40) & (m["Year"] >= 2000)][["country", "Year", "inflation"]]
print("values above 40% since 2000:")
print(flagged.round(1).to_string(index=False) if len(flagged) else "none")

w = m[m["iso"].isin(PEERS) & m["Year"].between(2000, 2023)].copy()
w["co2_kg_per_usd"] = w["co2"] * 1e9 / w["gdp_usd"]
w.loc[flagged.index, "inflation"] = np.nan

def cagr(s):
    return ((s.iloc[-1] / s.iloc[0]) ** (1 / (len(s) - 1)) - 1) * 100

peer = w.groupby("country").agg(mean_inflation=("inflation", "mean"), inflation_sd=("inflation", "std"),
                                real_growth_cagr=("gdp_pc_real", cagr)).round(1)
last = w[w["Year"] == 2023].set_index("country")
peer["gdp_pc_2023"] = last["gdp_pc"].round(0)
peer["co2_per_person_2023"] = last["co2_per_capita"].round(2)
peer.sort_values("real_growth_cagr", ascending=False)

## Step 6 · Charts

In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(k.index, k["gdp_pc"], color=GREY, lw=1.8, label="Current US$")
ax[0].plot(k.index, k["gdp_pc_real"], color=BLUE, lw=2.2, label="Deflated by US prices")
ax[0].set_title("Kenya GDP per person, 1960-2023"); ax[0].set_ylabel("US$ per person"); ax[0].legend(frameon=False)
pp = peer.sort_values("real_growth_cagr")
ax[1].barh(pp.index, pp["real_growth_cagr"], color=[BLUE if c == "Kenya" else GREEN for c in pp.index])
ax[1].set_title("Real GDP per person growth, 2000-2023"); ax[1].set_xlabel("% per year")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2)
kk = k.loc[1980:]
ax[0].bar(kk.index, kk["inflation"], color=[AMBER if v >= 15 else BLUE for v in kk["inflation"]])
ax[0].set_title("Kenya annual inflation (amber = 15% or more)"); ax[0].set_ylabel("% per year")
ax[1].scatter(peer["mean_inflation"], peer["inflation_sd"], s=60, color=[BLUE if c == "Kenya" else GREY for c in peer.index])
for name, r in peer.iterrows():
    ax[1].annotate(name, (r["mean_inflation"], r["inflation_sd"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax[1].set_title("Inflation level vs volatility, 2000-2023")
ax[1].set_xlabel("Mean inflation (%)"); ax[1].set_ylabel("Standard deviation (points)")
plt.tight_layout(); plt.show()

## Step 7 · Key facts, and what it means for lending

**What:** state the key facts with ranks, then a simple lending illustration: a loan at a fixed 16% (an illustrative rate, not a market rate) earns 16% minus inflation in real terms. What nominal rate keeps a 6-point real margin at typical and at bad inflation?

In [ ]:
kw = k.loc[2000:2023]
rank = peer.rank(ascending=False)
ci = w.pivot(index="Year", columns="country", values="co2_kg_per_usd")
co2_change = (ci.loc[2023, "Kenya"] / ci.loc[2000, "Kenya"] - 1) * 100
print(f"- GDP per person: US${k.loc[2000, 'gdp_pc']:,.0f} in 2000 -> US${k.loc[2023, 'gdp_pc']:,.0f} in 2023.")
print(f"- Real growth {peer.loc['Kenya', 'real_growth_cagr']:.1f}% a year, rank {int(rank.loc['Kenya', 'real_growth_cagr'])} of 7.")
print(f"- Mean inflation {kw['inflation'].mean():.1f}% (rank {int(rank.loc['Kenya', 'mean_inflation'])} of 7, 1 = highest); "
      f"peak {kw['inflation'].max():.1f}% in {kw['inflation'].idxmax()}; years at 10% or more: {int((kw['inflation'] >= 10).sum())} of {len(kw)}.")
print(f"- CO2 per US$ of GDP changed by {co2_change:.0f}% from 2000 to 2023.")

RATE, TARGET = 16.0, 6.0
real_yield = RATE - kw["inflation"]
print(f"\nA fixed {RATE:.0f}% loan earned a real yield of {real_yield.min():.1f} to {real_yield.max():.1f} points; "
      f"years below zero: {int((real_yield < 0).sum())}")
for q in (0.5, 0.9):
    pi = kw["inflation"].quantile(q)
    print(f"To keep {TARGET:.0f} points above inflation at the {int(q * 100)}th percentile ({pi:.1f}%): about {pi + TARGET:.1f}% nominal")

**Robustness check.** Rankings depend on the window. Over 2010–2023 only:

In [ ]:
recent = w[w["Year"].between(2010, 2023)].groupby("country")["inflation"].std().round(1).sort_values(ascending=False)
print("Kenya's volatility rank, 2010-2023:", int(recent.rank(ascending=False)["Kenya"]), "of", len(recent))

## Key findings and recommendations

- Kenya's GDP per person rose from about **US$415 (2000) to US$1,941 (2023)**; real growth of **4.6% a year is 2nd of 7**.
- Mean inflation of **8.0% is mid-table**, but it peaked at 27.7% in 2009, and 4 of 24 years were at 10% or more.
- *Price loans for inflation volatility, not only average inflation, and stress-test affordability in high-inflation years.*
- *Compare countries on a shared time window:* over 2010–2023, Kenya's volatility rank changes.

**Limitations.** Annual national figures, deflated by US prices, with no borrower data. These are questions to test with current data, not lending decisions.